# PriX FSS / Wavelet-LUT GPU Benchmark — Wave vs. SIGMA vs. Curl vs. SHAFT (Google Colab)

Make sure **Runtime → Change runtime type → GPU** (T4 / L4 / A100) before running.

This notebook runs [`benchmarks/benchmark_fss_modes.py`](https://github.com/mugurbil/prix/blob/main/benchmarks/benchmark_fss_modes.py) on the Colab GPU with **2 MPC parties on the same device** and reports, per primitive / sublayer / Transformer model:

| Column | Meaning |
|---|---|
| `rounds` | online communication rounds |
| `online B` / `online GB 1P` | online bytes sent by **one** party |
| `key B` / `key GB 1P` | **preprocessing key size** one party would receive from a dealer (PriX 62-bit seeds; `@l128` re-counts seed correction words at $\lambda=128$, i.e. 16 B / DPF level like SIGMA / Orca) |
| `time` | online wall-clock on the GPU |
| `MAE` | accuracy vs. float64 |

Variants:

| key | config | protocol |
|---|---|---|
| `wave_det` | `mode="wave"`, `trunc_method.lut="wave"` | 2-round $\text{bior}(5,3)$ LUT; $(J+\mathfrak{j})$-level Grotto DPF + $\mathfrak{j}$-level DCF (deterministic) |
| `wave_sigma_tr` | `mode="wave"`, `trunc_method.lut="sigma"` | 3-round $\text{bior}(5,3)$ LUT; SIGMA 1-round $\Pi^{\text{TR}}$ (single-path $\mathfrak{j}$-level DCF) + $J$-level DPF (deterministic) |
| `wave_prob` | `mode="wave"`, `trunc_method.lut="wave_prob"` | 2-round $\text{bior}(5,3)$ LUT; 0-round local probabilistic split + $J$-level DPF |
| `sigma` | `mode="sigma"` | SIGMA (PoPETs 2024) 0th-order Haar LUT protocols |
| `curl` | `mode="lut"` | Curl 4-round $\text{bior}(5,3)$ LUTs (pre-expanded one-hot + EGK truncation) |
| `shaft` | `mode="shaft"` | SHAFT Fourier activations + ODE Softmax |

> **Caveat (please read).** All numbers are **online-phase** figures with a *simulated* dealer (`TFP`): both parties expand the same PRG seeds, so **no key material is transferred** and the dealer's `Gen` runs inline on the online path. The `key B` columns report what a real dealer would have to send. SIGMA's published GPU latencies *include* reading pre-generated keys and copying them host→device, so they are **not** directly comparable to the `time` column here; compare rounds, online bytes, key bytes and accuracy instead, and treat GPU compute time as a device-local measurement only.

In [ ]:
!nvidia-smi
!pip install -q -U "jax[cuda12]" pywavelets pyyaml scipy cloudpickle
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert jax.default_backend() == "gpu", "Select a GPU runtime (Runtime -> Change runtime type -> GPU)"

In [ ]:
# Clone or update PriX
!if [ -d /content/prix ]; then git -C /content/prix pull; else git clone https://github.com/mugurbil/prix.git /content/prix; fi
!pip install -q -e /content/prix

In [ ]:
# 1) Primitives (N = 100k elements) + BERT-base sublayers on the GPU, all six variants.
#    (~5-10 min on a T4; use --n-prim 1000000 on an A100 to see the large-N regime.)
!cd /content/prix && python benchmarks/benchmark_fss_modes.py --device cuda \
    --modes wave_det,wave_sigma_tr,wave_prob,sigma,curl,shaft \
    --models "" --n-prim 100000 --repeats 5 \
    --out /content/fss_modes_gpu_primitives.json

In [ ]:
# 2) End-to-end Transformer blocks (BERT-base / BERT-large / GPT-2 @ L=64), extrapolated to all layers.
!cd /content/prix && python benchmarks/benchmark_fss_modes.py --device cuda \
    --modes wave_det,wave_sigma_tr,wave_prob,sigma,curl,shaft \
    --models bert_base,bert_large,gpt2_64 --skip-primitives --skip-sublayers --repeats 3 \
    --out /content/fss_modes_gpu_models.json

In [ ]:
# 3) Summary tables (pandas)
import json, pandas as pd

prim = json.load(open("/content/fss_modes_gpu_primitives.json"))
mod = json.load(open("/content/fss_modes_gpu_models.json"))
env = prim.pop("_env"); mod.pop("_env", None)
print("GPU:", env.get("gpu"), "| JAX", env["jax"])

rows = []
for key, r in prim.items():
    for name, st in r["primitives"].items():
        rows.append({"variant": key, "primitive": name, "rounds": st["rounds"],
                     "online B/elem": round(st["online_bytes_per_elem"], 1),
                     "key B/elem": round(st["key_bytes_per_elem"], 1),
                     "key B/elem @l128": round(st["key_bytes_per_elem_l128"], 1),
                     "time ms": round(st["time_ms"], 2), "MAE": st["mae"], "max err": st["max_err"]})
df_prim = pd.DataFrame(rows).set_index(["primitive", "variant"]).sort_index()
display(df_prim)

rows = []
for key, r in prim.items():
    for name, st in r["sublayers"].items():
        rows.append({"variant": key, "sublayer": name, "rounds": st["rounds"],
                     "online MB 1P": round(st["online_bytes_1p"] / 1e6, 2),
                     "key MB 1P": round(st["key_bytes_1p"] / 1e6, 1),
                     "key MB 1P @l128": round(st["key_bytes_1p_l128"] / 1e6, 1),
                     "time s": round(st["time_s"], 3), "MAE": st["mae"]})
display(pd.DataFrame(rows).set_index(["sublayer", "variant"]).sort_index())

rows = []
for key, r in mod.items():
    for name, st in r["models"].items():
        rows.append({"variant": key, "model": name, "rounds (total)": st["total_rounds"],
                     "online GB 1P": round(st["total_online_gb_1p"], 3),
                     "online GB 2P": round(st["total_online_gb_2p"], 3),
                     "key GB 1P": round(st["total_key_gb_1p"], 2),
                     "key GB 1P @l128": round(st["total_key_gb_1p_l128"], 2),
                     "time s (all layers)": round(st["total_time_s"], 2)})
display(pd.DataFrame(rows).set_index(["model", "variant"]).sort_index())